# Лабораторная работа 1.4. Метод вращений

**Задание.** Реализовать метод вращений в виде программы, задавая в качестве входных данных матрицу
и точность вычислений. Используя разработанное ПО, найти собственные значения и собственные векторы
симметрических матриц. Проанализировать зависимость погрешности вычислений от числа итераций.

$$
A = \begin{pmatrix} 2 & 8 & 7 \\ 8 & 2 & 7 \\ 7 & 7 & -8 \end{pmatrix}
$$

## Реализация

In [1]:
import math


def off_norm(A):
    n = len(A)
    return math.sqrt(sum(A[i][j] ** 2 for i in range(n) for j in range(i + 1, n)))


def multiply(A, B):
    return [[sum(A[i][k] * B[k][j] for k in range(len(B))) for j in range(len(B[0]))]
            for i in range(len(A))]


def transpose(A):
    return [list(row) for row in zip(*A)]


def rotation(A, eps, max_iter=1000):
    n = len(A)
    for i in range(n):
        for j in range(i + 1, n):
            if abs(A[i][j] - A[j][i]) > 1e-12:
                raise ValueError("Матрица не симметрическая")

    A = [list(map(float, row)) for row in A]
    V = [[float(i == j) for j in range(n)] for i in range(n)]
    history = [off_norm(A)]
    diags = [[A[i][i] for i in range(n)]]

    k = 0
    while history[-1] > eps:
        if k >= max_iter:
            raise RuntimeError("Метод вращений не сошёлся за max_iter итераций")
        
        i, j = max(((p, q) for p in range(n) for q in range(p + 1, n)),
                   key=lambda pq: abs(A[pq[0]][pq[1]]))
        if A[i][i] == A[j][j]:
            phi = math.pi / 4
        else:
            phi = 0.5 * math.atan(2 * A[i][j] / (A[i][i] - A[j][j]))

        U = [[float(p == q) for q in range(n)] for p in range(n)]
        c, s = math.cos(phi), math.sin(phi)
        U[i][i], U[j][j] = c, c
        U[i][j], U[j][i] = -s, s

        A = multiply(multiply(transpose(U), A), U)
        V = multiply(V, U)
        k += 1
        history.append(off_norm(A))
        diags.append([A[i][i] for i in range(n)])

    return [A[i][i] for i in range(n)], V, k, history, diags


def show(M, name=None, prec=6):
    if name:
        print(f"{name} =")
    for row in M:
        print("  [" + " ".join(f"{(0.0 if abs(v) < 1e-12 else v):{prec + 6}.{prec}f}" for v in row) + " ]")
    print()

## Исходные данные

In [2]:
A = [
    [2, 8,  7],
    [8, 2,  7],
    [7, 7, -8],
]
eps = 1e-6

show(A, "A", prec=0)
print(f"eps = {eps}")

A =
  [     2      8      7 ]
  [     8      2      7 ]
  [     7      7     -8 ]

eps = 1e-06


## Собственные значения и собственные векторы

In [3]:
lams, V, k, hist, _ = rotation(A, eps)

print(f"Итераций: {k},  t(A^(k)) = {hist[-1]:.2e}\n")
for idx, lam in enumerate(lams, 1):
    v = [V[r][idx - 1] for r in range(len(A))]
    print(f"lambda_{idx} = {lam:12.6f}    x_{idx} = [" + ", ".join(f"{c:9.6f}" for c in v) + " ]")
print()
show(V, "U (собственные векторы по столбцам)")

Итераций: 2,  t(A^(k)) = 2.22e-15

lambda_1 =    14.379088    x_1 = [ 0.646663,  0.646663,  0.404542 ]
lambda_2 =    -6.000000    x_2 = [-0.707107,  0.707107,  0.000000 ]
lambda_3 =   -12.379088    x_3 = [-0.286054, -0.286054,  0.914519 ]

U (собственные векторы по столбцам) =
  [    0.646663    -0.707107    -0.286054 ]
  [    0.646663     0.707107    -0.286054 ]
  [    0.404542     0.000000     0.914519 ]



## Анализ зависимости погрешности от числа итераций

In [4]:
def error_table(A):
    lams_ref, _, _, hist, diags = rotation(A, 1e-14)
    lams_ref = sorted(lams_ref)
    print(f"{'k':>3} | {'t(A^(k))':>10} | {'t_k / t_(k-1)':>13} | {'max|a_ii - lambda_i|':>20}")
    print("-" * 57)
    for k, (t, d) in enumerate(zip(hist, diags)):
        ratio = f"{t / hist[k - 1]:13.3e}" if k > 0 else " " * 13
        err = max(abs(x - y) for x, y in zip(sorted(d), lams_ref))
        print(f"{k:3} | {t:10.3e} | {ratio} | {err:20.3e}")


def iterations_table(A):
    print(f"{'eps':>8} | {'итераций':>8} | {'t(A^(k))':>10}")
    print("-" * 33)
    for p in range(1, 13):
        e = 10.0 ** -p
        _, _, k_e, h_e, _ = rotation(A, e)
        print(f"{e:8.0e} | {k_e:8} | {h_e[-1]:10.3e}")


error_table(A)

  k |   t(A^(k)) | t_k / t_(k-1) | max|a_ii - lambda_i|
---------------------------------------------------------
  0 |  1.273e+01 |               |            1.238e+01
  1 |  9.899e+00 |     7.778e-01 |            4.379e+00
  2 |  2.220e-15 |     2.243e-16 |            0.000e+00
